# EDA: Population Allele Frequency Data (Reference)

Sources: `gme_1000genomes_freq/*.xlsx` (four key SNPs) and `saudi_pgx_ngs/S1_table.xlsx` (Saudi PGx NGS allele counts with its own global-population comparison columns).

Each `gme_1000genomes_freq` file has two sheets: `variant_info` (a near-empty one-row summary, `MAF` is NaN there) and `population_frequencies` (the actual per-population allele counts). Read `population_frequencies` explicitly, and cross-check against `S1_table.xlsx`'s own AFR/AMR/ASJ/EAS/FIN/NFE/SAS columns, which are independently sourced gnomAD figures for the same variants.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", 100)
DATA_DIR = Path("..").resolve()
DECISIONS_PATH = Path("dataset_decisions.json")

def record_decision(name, verdict, reason, n_rows=None, n_cols=None, saved_to=None):
    decisions = {}
    if DECISIONS_PATH.exists():
        decisions = json.loads(DECISIONS_PATH.read_text())
    decisions[name] = {
        "verdict": verdict,
        "reason": reason,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "saved_to": saved_to,
    }
    DECISIONS_PATH.write_text(json.dumps(decisions, indent=2))
    print(f"Recorded decision for '{name}': {verdict}")


In [ ]:
import glob
freq_files = sorted(glob.glob(str(DATA_DIR / "usable/gme_1000genomes_freq/*.xlsx")))
pop_freqs = {}
for f in freq_files:
    rsid = Path(f).stem
    d = pd.read_excel(f, sheet_name="population_frequencies")
    pop_freqs[rsid] = d
    print(f"{rsid}: {d.shape[0]} population rows")
pop_freqs["rs9923231"].head(10)

## Which gene/variant does each rsID correspond to?

In [ ]:
rsid_gene_map = {
    "rs9923231": "VKORC1 -1639G>A (the main VKORC1 dosing variant)",
    "rs1799853": "CYP2C9*2 (R144C)",
    "rs1057910": "CYP2C9*3 (I359L)",
    "rs2108622": "CYP4F2*3 (V433M)",
}
for rsid, desc in rsid_gene_map.items():
    print(rsid, "->", desc)

## Cross-check: gnomAD population MAF (from gme_1000genomes_freq) vs the same figures cited in the Saudi S1 table

`S1_table.xlsx` reports its own AFR/AMR/ASJ/EAS/FIN/NFE/SAS allele-frequency columns for CYP2C9 variants alongside the Saudi cohort's MAF. If our independently-pulled gnomAD numbers roughly match those columns, that is an internal consistency check on the systematic review's cited figures.

## File inventory: `saudi_pgx_ngs/` (5 supplementary tables, S1 to S5)

**Used:** `S1_table.xlsx` (82 rows), a star-allele-level defining-variant table for 9 pharmacogenes (CYP2C9, CYP2C19, CYP3A5, CYP4F2, VKORC1, DPYD, TPMT, NUDT15) with Saudi MAF vs global population (AFR/AMR/EAS/NFE/SAS/Jordan/Qatari/GME/Iranome) comparisons. This is the cleanest table for the star-allele-level MAF comparison this notebook does.

**Not used:**
- `S4_table.xlsx` (10,010 rows): NGS amplicon panel design metadata (`amp_id`, insert coordinates, `gene_pool`), not allele-frequency data.
- `S5_table.xlsx` (4 rows): reference table of variant-effect-prediction algorithm thresholds (SIFT/PolyPhen/etc), a methods table, not data.
- `S2_table.xlsx` (64 rows) and `S3_table.xlsx` (46 rows): checked below, both are catalogs of rare/novel variants discovered by NGS in the Saudi cohort (all MAF under 0.5%, dbSNP IDs in the `rs3xxxxxxxx`+ range, annotated with SIFT/PolyPhen/MetaSVM/CADD pathogenicity scores). This is a different research question (novel-variant discovery) from S1's star-allele MAF comparison. Neither table contains any of the four warfarin-dosing variants used in this notebook (rs9923231, rs1799853, rs1057910, rs2108622), confirmed by exact RSID match below, so they add nothing to the population-MAF cross-check done here.

In [ ]:
target_rsids = list(rsid_gene_map.keys())
s2 = pd.read_excel(DATA_DIR / "usable/saudi_pgx_ngs/S2_table.xlsx")
s3 = pd.read_excel(DATA_DIR / "usable/saudi_pgx_ngs/S3_table.xlsx")

s2_hits = s2[s2["RSID"].isin(target_rsids)]
print("S2 rows matching our 4 target rsIDs:", len(s2_hits))
print("S3 has no RSID column; inspected manually. Its 46 rows are all rare or novel missense, splice,\n"
      "frameshift, or stop-gained variants in the same 7-gene panel (CYP2C9, CYP2C19, CYP3A5, CYP4F2,\n"
      "VKORC1, DPYD, TPMT), none of which are our 4 known defining variants.")
print()
print("S2/S3 MAF range (rare/novel variants, unlike S1's common defining alleles):")
print(pd.concat([s2["MAF SA"], s3["MAF SA"]]).describe()[["min", "max", "mean"]])

In [ ]:
s1 = pd.read_excel(DATA_DIR / "usable/saudi_pgx_ngs/S1_table.xlsx")
s1_cyp2c9_2 = s1[s1["Defining variants"] == "rs1799853"]
s1_cyp2c9_3 = s1[s1["Defining variants"] == "rs1057910"]
print("S1 row for CYP2C9*2 (rs1799853):")
print(s1_cyp2c9_2[["Allele", "MAF  Saudi", "Allele Frequency, EAS ", "Allele Frequency, NFE "]].to_string())
print()
print("S1 row for CYP2C9*3 (rs1057910):")
print(s1_cyp2c9_3[["Allele", "MAF  Saudi", "Allele Frequency, EAS ", "Allele Frequency, NFE "]].to_string())

In [ ]:
def gnomad_freq(rsid, pop_code, allele):
    d = pop_freqs[rsid]
    row = d[(d["population"] == f"gnomADg:{pop_code}") & (d["allele"] == allele)]
    return row["frequency"].iloc[0] if len(row) else None

print("rs1799853 (CYP2C9*2) gnomAD EAS frequency, minor allele:", gnomad_freq("rs1799853", "eas", "T"))
print("rs1057910 (CYP2C9*3) gnomAD EAS frequency, minor allele:", gnomad_freq("rs1057910", "eas", "C"))

(Exact allele-letter/population-code matching between the two sources needs care. S1's "EAS"/"NFE" labels and our `gnomADg:eas`/`gnomADg:nfe` rows are the same gnomAD population groups, but confirming the reference vs alternate allele orientation for each rsID individually is a manual pharmacogenomics check, not automated here. Flagging this as a to-do rather than asserting equality.)

## Verdict

**KEEP: reference/context only.** Useful for a discussion of population-specific allele-frequency variability (Saudi vs global, and consistency-checking the S1 table's own cited figures), but not patient-level rows, so still excluded from the supervised modeling pipeline.

In [ ]:
record_decision(
    "population_allele_freq",
    verdict="KEEP (reference)",
    reason="Population-level MAF context (1000 Genomes population_frequencies sheet plus Saudi NGS cohort), not per-patient data; excluded from the modeling pipeline.",
    n_rows=None, n_cols=None, saved_to=None,
)